# Phase 3 — Skin-Condition Classifier
## Google SCIN + Fitzpatrick17k via HuggingFace

**No Google Drive needed for data.** Both datasets load directly from HuggingFace.

Runtime -> Change runtime type -> **T4 GPU**

### One-time setup:
1. Accept SCIN terms at https://huggingface.co/datasets/google/scin
2. Accept Fitzpatrick17k terms (search `mattgroh/fitzpatrick17k` on HF)
3. Create a HF **read token** at https://huggingface.co/settings/tokens
4. Push your repo to GitHub, fill `GITHUB_USER` in Cell 1

| Source | Total | After face filter | Fitzpatrick |
|---|---|---|---|
| `google/scin` | ~10k | ~1-3k | yes |
| `fitzpatrick17k` | ~16k | ~2-4k | yes |
| **Combined** | **~26k** | **~3-7k** | **yes** |

## 0. Confirm GPU

In [ ]:
!nvidia-smi | head -12
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Clone repo + install
`ml_training` now lists `datasets` and `huggingface-hub` as hard deps.

In [ ]:
GITHUB_USER = "moodykhalif23"
BRANCH = "main"

!git clone -b {BRANCH} https://github.com/moodykhalif23/facemeup.git /content/skincare
%cd /content/skincare/backend_v2

!pip install -q -e ml_service
!pip install -q -e ml_training
print("install done")

## 2. HuggingFace login
Paste your read token from https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login()

# Non-interactive:
# import os; os.environ["HUGGINGFACE_TOKEN"] = "hf_YOUR_TOKEN"

## 3. Preview SCIN schema + body-part distribution
Streams 200 rows for the schema check, then the full set for the body-part count.

In [ ]:
from datasets import load_dataset
from collections import Counter

print("--- SCIN schema (first 200 rows) ---")
preview = load_dataset("google/scin", split="train[:200]")
for col, feat in preview.features.items():
    sample = "<image>" if col in ("image","img") else str(preview[0].get(col,""))[:60]
    print(f"  {col:<35} {str(feat):<25}  e.g. {sample!r}")

BP_CANDS   = ["body_part","anatom_site","location","site","region","body_location"]
FACE_TERMS = {"face","neck","scalp","cheek","forehead","chin","nose","perioral","periorbital","head"}
bp_col = next((c for c in preview.features if c.lower() in BP_CANDS), None)

if bp_col:
    print(f"\nbody-part column: {bp_col!r} -- loading full dataset...")
    ds_full = load_dataset("google/scin", split="train")
    dist = Counter(str(r.get(bp_col,"")).lower().strip() for r in ds_full)
    for val, n in dist.most_common(25):
        flag = "KEEP" if any(t in val for t in FACE_TERMS) else "skip"
        print(f"  [{flag}]  {val!r:<40} {n}")
    kept  = sum(n for v,n in dist.items() if any(t in v for t in FACE_TERMS))
    total = sum(dist.values())
    print(f"\n-> Face-region: ~{kept} / {total} ({100*kept/total:.0f}%) will enter precompute")
else:
    print("No body-part column found -- face detector will be the only filter.")

## 4. Precompute aligned face tensors

Streams from HuggingFace, filters to face/neck/scalp, runs:
`decode -> face-detect -> align -> CLAHE+WB -> save .npy`

| --source | What loads |
|---|---|
| `scin_hf` | SCIN only (~10 min) |
| `fitzpatrick17k` | Fitzpatrick17k only |
| `all` | Both combined (recommended, ~30-50 min) |

Fully resumable -- re-run to continue.

In [ ]:
# Cache HF downloads. Mount Drive to persist across sessions:
# from google.colab import drive; drive.mount('/content/drive')
# HF_CACHE = '/content/drive/MyDrive/skincare/hf_cache'
HF_CACHE = '/content/hf_cache'   # in-session only

In [ ]:
SOURCE = 'all'   # scin_hf | fitzpatrick17k | all

!python -m skin_training.data.precompute \
  --source {SOURCE} \
  --output /content/work \
  --aligned-size 256 \
  --hf-cache-dir {HF_CACHE} \
  --verbose 2>&1 | tail -50

In [ ]:
import json, re, pandas as pd
idx = json.loads(open('/content/work/index.json').read())
print(json.dumps(idx, indent=2))
n, pct = idx['n_samples'], idx.get('face_detect_skip_pct','?')
print(f"\nAligned face tensors: {n}  |  face-detector skip rate: {pct}%")
df = pd.read_csv('/content/work/labels.csv')
cond_cols = [c for c in df.columns if re.match(r'^c\d+_', c)]
print("\nCondition coverage:")
for col in cond_cols:
    n_pos = int(df[col].sum())
    pct2  = 100 * n_pos / max(1, len(df))
    flag  = "[!] " if n_pos < 30 else "    "
    print(f"  {flag}{col:<28} {n_pos:>6} positives  ({pct2:.1f}%)")
if n < 300:
    print("\n[!] Very few samples -- try --all-body-parts")

## 5. Configure training

In [ ]:
import yaml
from pathlib import Path
cfg = yaml.safe_load(Path('ml_training/configs/base.yaml').read_text())
cfg['data']['aligned_dir']          = '/content/work'
cfg['train']['checkpoint_dir']      = '/content/work/runs/exp1'
cfg['train']['batch_size']          = 32
cfg['train']['epochs']              = 25
cfg['train']['mixed_precision']     = True
cfg['train']['early_stop_patience'] = 6
Path('/content/config.yaml').write_text(yaml.safe_dump(cfg))
print(yaml.safe_dump(cfg))

## 6. Train (~40-80 min on T4)
Resumable -- interrupt and re-run to continue from last.pt.

In [ ]:
!python -m skin_training.train.loop --config /content/config.yaml --verbose

## 7. TensorBoard (run in a second tab while training)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/work/runs/exp1/tb

## 8. Export ONNX + roundtrip check

In [ ]:
!python -m skin_training.export.to_onnx \
  --checkpoint /content/work/runs/exp1/best.pt \
  --config /content/config.yaml \
  --output /content/work/skin_classifier_mobilenet.onnx \
  --verbose

## 9. Save to Drive + download
Drive only used here (output artifacts), never for input data.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/skincare/artifacts
!cp /content/work/skin_classifier_mobilenet.onnx /content/drive/MyDrive/skincare/artifacts/
!cp /content/work/runs/exp1/best.pt            /content/drive/MyDrive/skincare/artifacts/
!cp /content/work/index.json                   /content/drive/MyDrive/skincare/artifacts/
print('saved to Drive')
files.download('/content/work/skin_classifier_mobilenet.onnx')

## 10. Local install (Windows machine)

```powershell
Copy-Item $HOME\Downloads\skin_classifier_mobilenet.onnx `
  C:\Users\Sozuri\skincare\backend_v2\ml_service\models\

cd C:\Users\Sozuri\skincare\backend_v2
docker compose build ml-service
docker compose up -d ml-service

curl http://localhost:8013/healthz   # expect models_loaded: true
```

After that POST /api/v1/analyze returns "inference_mode": "onnx_mobilenet".

**Report back with:**
- Final val_F1_macro
- Fitzpatrick-stratified F1 block
- index.json (samples per source)

-> I will start Phase 6: MobileNet distillation + INT8 quant